# Michaelis–Menten enzyme kinetics

*biophysics · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/biophysics/michaelis_menten)

An enzyme binds its substrate, forms a complex and releases the product: $E + S \rightleftharpoons ES \to E + P$. The full mass-action equations are solved for three enzymes and a whole substrate titration at once, in one $(4, B, C)$ tensor, and compared with the quasi-steady-state rate law $v = V_{max} S/(K_m + S)$ and a Lineweaver–Burk fit.

**Vector form**

$$E + S \;\underset{k_{-1}}{\overset{k_1}{\rightleftharpoons}}\; ES \;\overset{k_2}{\longrightarrow}\; E + P,\qquad \frac{d[ES]}{dt} = k_1 [E][S] - (k_{-1} + k_2)[ES],\qquad v = \frac{dP}{dt} \approx \frac{V_{max}\,[S]}{K_m + [S]}$$

**Index (tensor) form**

$$\dot S_{bc} = -k^{1}_b E_{bc} S_{bc} + k^{-1}_b X_{bc} \qquad \dot X_{bc} = k^{1}_b E_{bc} S_{bc} - (k^{-1}_b + k^{2}_b)\,X_{bc} \qquad \dot P_{bc} = k^{2}_b X_{bc} \qquad A_{ck}A_{cl}\,a_{bl} = A_{ck}\,\frac{1}{v_{bc}}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $k^{1}_b,\ k^{-1}_b,\ k^{2}_b$ | `k1, k_1, k2` | (B,) | binding, unbinding and catalytic rate constants of enzyme $b$ ($k_2 = k_{cat}$) |
| $S^0_c$ | `S0` | (C,) | starting substrate concentration $c$ of the titration (µM) |
| $E_{bc},\ S_{bc},\ X_{bc},\ P_{bc}$ | `y[0], y[1], y[2], y[3]` | (4, B, C) | free enzyme, substrate, complex $X = ES$ and product in every test tube |
| $k^{1}_b E_{bc} S_{bc}$ | `einsum('b,bc,bc->bc', k1, E, S)` | (B, C) | no index summed: $k_b$ is broadcast over every concentration $c$ |
| $\dot y_{bc}$ | `rhs(y, k1, k_1, k2)` | (4, B, C) | right-hand side for RK4; $E + X = E_0$ and $S + X + P = S_0$ stay constant |
| $K_{m,b},\ V_{max,b}$ | `(k_1 + k2) / k1, k2 * E0` | (B,) | Michaelis constant (half-speed substrate level) and maximal rate |
| $v_{bc}$ | `k2 * ES.amax(0)` | (B, C) | initial (steady-state) velocity, measured after the short burst |
| $A_{ck}A_{cl}$ | `einsum('ck,cl->kl', A, A)` | (2, 2) | Lineweaver–Burk normal equations, $A_{ck} = [1/S_c,\ 1]$; repeated $c$ ⇒ summed |
| $a_{bl}$ | `linalg.solve(G, r.T)` | (2, B) | slope $K_m/V_{max}$ and intercept $1/V_{max}$ for every enzyme |

Mass action alone gives four coupled ODEs. Michaelis and Menten (1913), in the form of Briggs and Haldane (1925), noticed that the complex $ES$ builds up within a fraction of a second and then changes only slowly, so one may set $d[ES]/dt \approx 0$. Together with $E + ES = E_0$ this gives $[ES] = E_0 S/(K_m + S)$ and the rate law $v = k_2 [ES] = V_{max} S/(K_m + S)$ with $K_m = (k_{-1} + k_2)/k_1$ and $V_{max} = k_2 E_0$. The dots in the middle panel show every test tube's current $(S, k_2 [ES])$: after the burst they sit on the dashed curve and slide down it as the substrate is used up, which is the quasi-steady-state approximation (QSSA) at work. The reduced model $dS/dt = -V_{max} S/(K_m + S)$ (dashed orange) even has an exact solution, $S_0 - S + K_m \ln(S_0/S) = V_{max} t$. The approximation holds when $E_0 \ll S_0 + K_m$; raise $E_0$ to 2 µM and watch the full kinetics lag behind the QSSA because the substrate locked in the complex is no longer negligible. Biochemists measure the initial velocity $v_0$ at several $S_0$ and fit $1/v_0 = (K_m/V_{max})(1/S_0) + 1/V_{max}$, a straight line (Lineweaver–Burk, 1934): the $y$-intercept is $1/V_{max}$ and the $x$-intercept is $-1/K_m$. The double-reciprocal plot amplifies errors at low $S_0$, so with noisy data a direct nonlinear fit is better. Enzymes A, B and C trade binding against speed: A has a small $K_m$, C the largest $k_{cat}$; the catalytic efficiency $k_{cat}/K_m$ decides which wins at low substrate. Simplifications: one active site, irreversible product release, no inhibition and a well-mixed volume.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/biophysics/michaelis_menten/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    names = tensors.enzyme_names()
    k1, k_1, k2 = tensors.rate_constants(dev)  # (B,) each
    S0 = tensors.substrate_concentrations(dev)  # (C,) µM
    E0 = params["E0"]
    y0 = tensors.initial_state(E0, S0, len(names))  # (4, B, C)

    K_m = equation.michaelis_constant(k1, k_1, k2)  # (B,)
    V_max = equation.max_velocity(k2, E0)  # (B,)
    times, full, S_qssa = solve.solve(y0, k1, k_1, k2, V_max, K_m, params["t_end"])

    S_v, v0 = solve.initial_rate(full, k2)  # (B, C) measured initial velocities and the S they were measured at
    K_fit, V_fit = equation.lineweaver_burk(S_v, v0)

    enzyme_total = full[:, 0] + full[:, 2]  # E + ES
    substrate_total = full[:, 1] + full[:, 2] + full[:, 3]  # S + ES + P
    P_qssa = S0 - S_qssa  # product of the reduced model
    gap = ((full[:, 3] - P_qssa).abs().amax(dim=(0, 2)) / S0.max()).tolist()  # (B,) worst gap, fraction of S0
    print(f"E0 = {E0:g} µM, S0 = {S0[0]:g} … {S0[-1]:g} µM, {len(times) - 1} RK4 steps of {times[1]:.4f} s")
    print(
        f"conservation: max |E + ES − E0| = {(enzyme_total - E0).abs().max():.1e} µM, "
        f"max |S + ES + P − S0| = {(substrate_total - S0).abs().max():.1e} µM"
    )
    for b, name in enumerate(names):
        print(
            f"enzyme {name}: K_m = {K_m[b]:5.2f} µM (Lineweaver–Burk {K_fit[b]:5.2f})   "
            f"V_max = {V_max[b]:.3f} µM/s (fit {V_fit[b]:.3f})   "
            f"full vs QSSA product: max gap {100 * gap[b]:.2f} % of S0"
        )
    e = names.index(params["enzyme"])
    return plot.animate(
        times, full, P_qssa, S0, S_v, v0, K_m, V_max, K_fit, V_fit, names, e, out_dir / "michaelis_menten.gif"
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/biophysics/michaelis_menten/equation.py
r"""Physics → tensor translation.

An enzyme E binds its substrate S, forms a complex ES and turns it into product P:

    E + S  ⇌  ES  →  E + P        rates k1 E S (binding), k₋₁ ES (unbinding), k2 ES (catalysis)

For a batch of enzymes b and a titration of substrate concentrations c, mass action gives

    \dot E_{bc}  = -k^1_b E_{bc} S_{bc} + (k^{-1}_b + k^2_b) ES_{bc}
    \dot S_{bc}  = -k^1_b E_{bc} S_{bc} +  k^{-1}_b ES_{bc}
    \dot ES_{bc} = +k^1_b E_{bc} S_{bc} - (k^{-1}_b + k^2_b) ES_{bc}
    \dot P_{bc}  =  k^2_b ES_{bc}

so E + ES = E0 (enzyme is recycled) and S + ES + P = S0 (substrate atoms are conserved).
No index is summed in the kinetics: every (b, c) pair is its own test tube; the einsum calls make the
broadcasting of k_b over c explicit. The Lineweaver–Burk fit sums over the concentration index c.
"""

import torch


def reaction_rates(E, S, ES, k1, k_1, k2) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Binding k1 E S, unbinding k₋₁ ES and catalysis k2 ES.   E, S, ES: (B, C);  k: (B,)  →  three (B, C)."""
    binding = torch.einsum("b,bc,bc->bc", k1, E, S)  # needs an enzyme AND a substrate to meet
    unbinding = torch.einsum("b,bc->bc", k_1, ES)  # the complex falls apart unchanged
    catalysis = torch.einsum("b,bc->bc", k2, ES)  # the complex releases product and free enzyme
    return binding, unbinding, catalysis


def rhs(y: torch.Tensor, k1, k_1, k2) -> torch.Tensor:
    """dy/dt for y of shape (4, B, C) = (E, S, ES, P)."""
    E, S, ES, _ = y
    v1, v_1, v2 = reaction_rates(E, S, ES, k1, k_1, k2)
    return torch.stack([-v1 + v_1 + v2, -v1 + v_1, v1 - v_1 - v2, v2])


def michaelis_constant(k1, k_1, k2) -> torch.Tensor:
    """K_m = (k₋₁ + k2) / k1: the substrate level at which the enzyme runs at half speed.   (B,)"""
    return (k_1 + k2) / k1


def max_velocity(k2: torch.Tensor, E0: float) -> torch.Tensor:
    """V_max = k2 E0 = k_cat E0: every enzyme busy all the time.   (B,)"""
    return k2 * E0


def mm_rate(S: torch.Tensor, V_max: torch.Tensor, K_m: torch.Tensor) -> torch.Tensor:
    """Quasi-steady-state rate v_bc = V_max,b S_bc / (K_m,b + S_bc).   S: (B, C) or (C,)  →  (B, C)."""
    S = S.expand(len(V_max), -1)
    return torch.einsum("b,bc->bc", V_max, S / (K_m[:, None] + S))


def qssa_rhs(S: torch.Tensor, V_max: torch.Tensor, K_m: torch.Tensor) -> torch.Tensor:
    """dS/dt = -V_max S / (K_m + S): the reduced (Michaelis–Menten) model, one equation per test tube."""
    return -mm_rate(S, V_max, K_m)


def integrated_mm_time(S: torch.Tensor, S0: torch.Tensor, V_max: torch.Tensor, K_m: torch.Tensor) -> torch.Tensor:
    """Integrated Michaelis–Menten law: the time to go from S0 down to S,  t = (S0 − S + K_m ln(S0/S)) / V_max."""
    return (S0 - S + K_m[:, None] * torch.log(S0 / S)) / V_max[:, None]


def lineweaver_burk(S: torch.Tensor, v: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    r"""Fit 1/v = (K_m/V_max)(1/S) + 1/V_max by least squares for every enzyme at once.   S, v: (B, C).

    Design matrix A_{bck} = [1/S_{bc}, 1] (B, C, 2) and data y_{bc} = 1/v_{bc}. The normal equations

        G_{bkl} a_{bl} = r_{bk},   G_{bkl} = A_{bck} A_{bcl},   r_{bk} = A_{bck} y_{bc}

    sum over the concentration index c. Returns (K_m, V_max), each (B,).
    """
    A = torch.stack([1.0 / S, torch.ones_like(S)], dim=-1)  # (B, C, 2)
    G = torch.einsum("bck,bcl->bkl", A, A)  # (B, 2, 2), c summed
    r = torch.einsum("bck,bc->bk", A, 1.0 / v)  # (B, 2), c summed
    slope, intercept = torch.linalg.solve(G, r).T  # a_bl: one small 2×2 system per enzyme
    return slope / intercept, 1.0 / intercept


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/biophysics/michaelis_menten/tensors.py
"""Build the concrete tensors: rate constants of each enzyme, substrate concentrations, initial state.

Units: concentrations in µM, times in s, so k1 is in µM⁻¹ s⁻¹ and k₋₁, k2 are in s⁻¹.
"""

import torch

# name, k1 (µM⁻¹ s⁻¹), k₋₁ (s⁻¹), k2 = k_cat (s⁻¹)   →   K_m = (k₋₁ + k2) / k1
ENZYMES = [
    ("A", 1.0, 1.0, 4.0),  # K_m =  5 µM, k_cat =  4 s⁻¹: binds tightly
    ("B", 0.5, 2.0, 3.0),  # K_m = 10 µM, k_cat =  3 s⁻¹: in between
    ("C", 0.5, 5.0, 10.0),  # K_m = 30 µM, k_cat = 10 s⁻¹: fast but weak binding
]

SUBSTRATE = [2.5, 5.0, 10.0, 20.0, 40.0, 80.0]  # S0 of the titration, µM


def enzyme_names() -> list[str]:
    return [name for name, *_ in ENZYMES]


def rate_constants(device: torch.device) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """k1, k₋₁, k2, each of shape (B,): one entry per enzyme b."""
    k = torch.tensor([rates for _, *rates in ENZYMES], device=device)  # (B, 3)
    return k[:, 0], k[:, 1], k[:, 2]


def substrate_concentrations(device: torch.device) -> torch.Tensor:
    """S0_c in µM, shape (C,)."""
    return torch.tensor(SUBSTRATE, device=device)


def initial_state(E0: float, S0: torch.Tensor, n_enzymes: int) -> torch.Tensor:
    """y0 of shape (4, B, C) = (E, S, ES, P): free enzyme and substrate only, no complex or product yet."""
    S = S0.expand(n_enzymes, -1).clone()  # (B, C): every enzyme gets the whole titration
    E = torch.full_like(S, E0)
    return torch.stack([E, S, torch.zeros_like(S), torch.zeros_like(S)])


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/biophysics/michaelis_menten/solve.py
"""Integrate the full mass-action kinetics and the reduced Michaelis–Menten model with RK4."""

import torch

from problems._common.integrators import integrate

from .equation import qssa_rhs, rhs


def time_step(k1, k_1, k2, S0: torch.Tensor, E0: float) -> float:
    """Binding relaxes at about k1 (S + E0) + k₋₁ + k2; keep dt below 1 / (fastest rate) for RK4."""
    k_fastest = (k1 * (S0.max() + E0) + k_1 + k2).max().item()
    return min(0.01, 1.0 / k_fastest)


def solve(y0: torch.Tensor, k1, k_1, k2, V_max, K_m, t_end: float):
    """Return (times (T,), full (T, 4, B, C), S of the reduced model (T, B, C)).

    All enzymes b and substrate concentrations c advance together in one tensor.
    """
    dt = time_step(k1, k_1, k2, y0[1], y0[0].max().item())
    n = int(round(t_end / dt))
    full = integrate(lambda y, t: rhs(y, k1, k_1, k2), y0, dt, n)  # E + S ⇌ ES → E + P
    reduced = integrate(lambda S, t: qssa_rhs(S, V_max, K_m), y0[1], dt, n)  # dS/dt = −V_max S/(K_m + S)
    times = torch.arange(n + 1, device=y0.device) * dt
    return times, full, reduced


def initial_rate(full: torch.Tensor, k2: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """v0_bc = k2 · max_t ES_bc: the steady-state velocity reached right after the short pre-steady-state burst.

    Returns (S at that moment, v0), both (B, C). A little substrate is already bound or used up by then,
    so the fit uses this S rather than S0 (the difference is a few % at the lowest S0).
    """
    ES_peak, i_peak = full[:, 2].max(dim=0)  # (B, C) value and time index of the peak
    S_peak = full[:, 1].gather(0, i_peak[None]).squeeze(0)  # S_bc at that time
    return S_peak, torch.einsum("b,bc->bc", k2, ES_peak)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/biophysics/michaelis_menten/plot.py
"""Animate enzyme kinetics: progress curves (full vs QSSA), the rate law v(S) and a Lineweaver–Burk plot."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

S_COLORS = ["#93c5fd", "#60a5fa", "#3b82f6", "#2563eb", "#1d4ed8", "#1e3a8a"]  # light → dark = low → high S0


def animate(
    times: torch.Tensor,
    full: torch.Tensor,
    P_qssa: torch.Tensor,
    S0: torch.Tensor,
    S_v: torch.Tensor,
    v0: torch.Tensor,
    K_m: torch.Tensor,
    V_max: torch.Tensor,
    K_fit: torch.Tensor,
    V_fit: torch.Tensor,
    names: list[str],
    e: int,
    path: Path,
) -> Path:
    stride = max(1, len(times) // 1500)  # RK4 takes small steps; ~1500 points are plenty for drawing
    t = times[::stride].cpu().numpy()
    S = full[::stride, 1].cpu().numpy()  # (T, B, C)
    ES = full[::stride, 2].cpu().numpy()
    P = full[::stride, 3].cpu().numpy()
    Pq = P_qssa[::stride].cpu().numpy()
    s0, s_v, v_init = S0.cpu().numpy(), S_v.cpu().numpy(), v0.cpu().numpy()
    Km, Vm, Kf, Vf = (x.cpu().numpy() for x in (K_m, V_max, K_fit, V_fit))
    k2 = Vm / full[0, 0, :, 0].cpu().numpy()  # k_cat = V_max / E0
    B, C = len(names), len(s0)
    colors = ["C0", "C1", "C2"]

    fig = plt.figure(figsize=(8.4, 4.4))
    grid = fig.add_gridspec(2, 2, width_ratios=[1.15, 1])
    ax_p = fig.add_subplot(grid[:, 0])
    ax_v = fig.add_subplot(grid[0, 1])
    ax_lb = fig.add_subplot(grid[1, 1])

    # --- progress curves of one enzyme: fraction of substrate turned into product ---
    ax_p.set_xlim(0, t[-1])
    ax_p.set_ylim(0, 1.05)
    ax_p.set_xlabel("time (s)")
    ax_p.set_ylabel("product formed  P / S₀")
    ax_p.set_title(f"enzyme {names[e]}: solid = full kinetics, dashed = QSSA", fontsize=10)
    full_lines = [ax_p.plot([], [], lw=1.8, color=S_COLORS[c], label=f"S₀ = {s0[c]:g} µM")[0] for c in range(C)]
    qssa_lines = [ax_p.plot([], [], "--", lw=1.2, color="#f59e0b")[0] for c in range(C)]
    ax_p.legend(loc="upper left", fontsize=7.5)
    clock = ax_p.text(0.97, 0.04, "", transform=ax_p.transAxes, ha="right", fontsize=9)

    # --- rate law: v = V_max S / (K_m + S) (lines) and the current (S, k2 ES) of every test tube (dots) ---
    S_fine = np.linspace(0, s0.max() * 1.05, 200)
    ax_v.set_xlim(0, S_fine[-1])
    ax_v.set_ylim(0, Vm.max() * 1.1)
    ax_v.set_xlabel("substrate S (µM)", labelpad=1)
    ax_v.set_ylabel("rate v (µM/s)")
    ax_v.set_title("rate law; dots = current (S, k₂·ES)", fontsize=10)
    for b in range(B):
        ax_v.plot(
            S_fine,
            Vm[b] * S_fine / (Km[b] + S_fine),
            "--",
            lw=2 if b == e else 1,
            color=colors[b],
            label=f"{names[b]}: Kₘ = {Km[b]:g} µM",
        )
    ax_v.legend(loc="upper left", fontsize=7)
    dots = [ax_v.plot([], [], "o", ms=3.5, color=colors[b])[0] for b in range(B)]

    # --- Lineweaver–Burk: 1/v against 1/S0 is a straight line; intercepts give 1/V_max and −1/K_m ---
    x = np.linspace(-1.3 / Km.min(), 1.1 / s0.min(), 50)
    for b in range(B):
        ax_lb.plot(1 / s_v[b], 1 / v_init[b], "o", ms=3.5, color=colors[b])
        ax_lb.plot(x, (Kf[b] * x + 1) / Vf[b], "-", lw=1, color=colors[b])
    ax_lb.axhline(0, color="#374151", lw=0.8)
    ax_lb.axvline(0, color="#374151", lw=0.8)
    ax_lb.set_xlim(x[0], x[-1])
    ax_lb.set_ylim(-0.1 * (1 / v_init).max(), 1.1 * (1 / v_init).max())
    ax_lb.set_xlabel("1 / S (1/µM)", labelpad=1)
    ax_lb.set_ylabel("1 / v₀ (s/µM)")
    ax_lb.set_title("Lineweaver–Burk fit (x-intercept = −1/Kₘ)", fontsize=10)

    def update(step: int):
        for c in range(C):
            full_lines[c].set_data(t[: step + 1], P[: step + 1, e, c] / s0[c])
            qssa_lines[c].set_data(t[: step + 1], Pq[: step + 1, e, c] / s0[c])
        for b in range(B):
            dots[b].set_data(S[step, b], k2[b] * ES[step, b])
        clock.set_text(f"t = {t[step]:.1f} s")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 90), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'E0': 0.1,
    't_end': 60.0,
    'enzyme': 'A'
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.biophysics.michaelis_menten")]:
    del sys.modules[name]
main = importlib.import_module("problems.biophysics.michaelis_menten.main")
problem_dir = Path("problems/biophysics/michaelis_menten")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
